# Lab 14: GPU Compute as a Commodity: From Compute Futures to Inference Markets

**Goal:** use live compute and inference prices to understand how financial markets can hedge physical-infrastructure risk — and why a hedge can break when the financial asset and the business exposure are not identical.

In this lab you will:

1. Query the Ornn Compute Price Index (OCPI) API for **H100 SXM** history.
2. Inspect how the market price of an H100 GPU-hour moves.
3. Model the revenue of an operator running a small H100 fleet.
4. Add a **cash-settled financial forward** on H100 compute prices.
5. See why a fixed-notional hedge still leaves physical utilization risk.
6. Connect standardized compute to a decentralized inference market.
7. Build a simple **inference intermediary** that buys inference from Darkbloom, sells fixed-price inference to a customer, and uses H100 compute futures as a proxy hedge.

The key question is:

> **Can financial markets turn volatile AI infrastructure costs into a stable product for customers? What happens when the hedge and the underlying business are related, but not identical?**

**Expected time:** 40–45 minutes.

**Data sources:** Ornn Compute Price Index (OCPI) and Darkbloom public inference pricing, queried live from their public APIs.


In [ ]:
import os
from urllib.parse import quote

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

# Environment overrides are only for instructor testing. Students should leave them unchanged.
ORNN_API_BASE = os.environ.get("ORNN_API_BASE", "https://api.ornnai.com")
DARKBLOOM_API_BASE = os.environ.get("DARKBLOOM_API_BASE", "https://api.darkbloom.dev")

GPU_NAME = "H100 SXM"

# Fleet assumptions used in the first half of the lab.
N_GPUS = 8
HORIZON_DAYS = 90
EXPECTED_UTILIZATION = 0.70

Q_MAX = N_GPUS * 24 * HORIZON_DAYS
Q_EXPECTED = Q_MAX * EXPECTED_UTILIZATION

print(f"Fleet: {N_GPUS} GPUs")
print(f"90-day maximum capacity: {Q_MAX:,.0f} GPU-hours")
print(f"Expected sold compute at {EXPECTED_UTILIZATION:.0%} utilization: {Q_EXPECTED:,.0f} GPU-hours")


## Part 1 — Query the Ornn H100 index

The API plumbing is supplied. Run the next cell to pull the public H100 SXM price history and inspect the response.

The student work in this lab is focused on **economic choices and interpretation**, not HTTP/request syntax.


In [ ]:
gpu_slug = quote(GPU_NAME, safe="")
url = f"{ORNN_API_BASE}/api/gpu/{gpu_slug}/index-history"

response = requests.get(url, timeout=20)

if response.status_code == 429:
    raise RuntimeError("Ornn API rate limit reached. Ask the instructor to rerun later.")
response.raise_for_status()

payload = response.json()

print("Source: Ornn Compute Price Index (OCPI)")
print("GPU:", payload.get("gpu_type"))
print("Access tier:", payload.get("access", "not reported"))
print("Observations returned:", len(payload.get("data", [])))


In [ ]:
# Validate the documented response shape before using it.
if payload.get("success") is not True:
    raise ValueError(f"Ornn API did not report success: {payload}")

if payload.get("gpu_type") != GPU_NAME:
    raise ValueError(f"Expected {GPU_NAME!r}, received {payload.get('gpu_type')!r}")

raw_rows = payload.get("data")
if not isinstance(raw_rows, list) or len(raw_rows) < 2:
    raise ValueError("Expected at least two daily OCPI observations.")

prices = pd.DataFrame(raw_rows)
required_cols = {"timestamp", "index_value"}
missing = required_cols - set(prices.columns)
if missing:
    raise ValueError(f"Missing expected fields from Ornn response: {sorted(missing)}")

prices = prices[["timestamp", "index_value"]].copy()
prices["timestamp"] = pd.to_datetime(prices["timestamp"], utc=True, errors="raise")
prices["index_value"] = pd.to_numeric(prices["index_value"], errors="raise")
prices = prices.dropna().drop_duplicates(subset="timestamp").sort_values("timestamp").reset_index(drop=True)

if (prices["index_value"] <= 0).any():
    raise ValueError("OCPI prices should be positive.")

display(prices.tail())


## Part 2 — What does the market price of compute look like?

OCPI is a benchmark for the transacted rental price of compute, expressed in **USD per GPU-hour**. The code below visualizes the H100 series and summarizes the range observed in the API response.


In [ ]:
latest_time = prices.loc[prices.index[-1], "timestamp"]
latest_price = float(prices.loc[prices.index[-1], "index_value"])

# Find the observation closest to 30 calendar days before the latest print.
target_30d = latest_time - pd.Timedelta(days=30)
idx_30d = (prices["timestamp"] - target_30d).abs().idxmin()
price_30d = float(prices.loc[idx_30d, "index_value"])
change_30d = latest_price / price_30d - 1

summary = pd.Series({
    "latest_$/GPU-hour": latest_price,
    "approx_30d_change_%": 100 * change_30d,
    "sample_min_$/GPU-hour": prices["index_value"].min(),
    "sample_median_$/GPU-hour": prices["index_value"].median(),
    "sample_max_$/GPU-hour": prices["index_value"].max(),
})

display(summary.to_frame("value"))

plt.figure(figsize=(9, 4))
plt.plot(prices["timestamp"], prices["index_value"])
plt.xlabel("Date")
plt.ylabel("USD per H100 SXM GPU-hour")
plt.title("Ornn Compute Price Index — H100 SXM")
plt.grid(alpha=0.2)
plt.show()


### Question 1 — Whose risk is this?

You own GPUs and rent their compute on demand. Which direction of H100 price movement hurts your revenue, and why?

**Your answer:**  
`WRITE HERE`


## Part 3 — You are the GPU operator

Suppose your 8-GPU fleet will operate for the next 90 days. The fleet has maximum capacity

$$
Q_{\max} = 8 \times 24 \times 90 = 17{,}280 \text{ GPU-hours}.
$$

You expect average utilization of 70%, so your expected **physical rental volume** is

$$
\hat{Q} = Q_{\max} \times 0.70 = 12{,}096 \text{ GPU-hours}.
$$

If the average H100 rental price over the period is $S$, expected rental revenue is

$$
R_{\text{spot}} = \hat{Q}S.
$$

To create market-based price scenarios without inventing arbitrary dollar values, we use the 10th, 50th, and 90th percentiles of the H100 prices returned by your API query.

In [ ]:
scenario_prices = prices["index_value"].quantile([0.10, 0.50, 0.90])
scenario_prices.index = ["Low (10th pct)", "Median (50th pct)", "High (90th pct)"]

unhedged = pd.DataFrame({
    "H100_price_$/GPU-hour": scenario_prices,
})
unhedged["expected_GPU-hours_sold"] = Q_EXPECTED
unhedged["unhedged_revenue_$"] = unhedged["H100_price_$/GPU-hour"] * Q_EXPECTED

display(unhedged)

plt.figure(figsize=(8, 4))
plt.bar(unhedged.index, unhedged["unhedged_revenue_$"])
plt.ylabel("90-day rental revenue ($)")
plt.xlabel("H100 price scenario")
plt.title("Unhedged operator revenue at 70% utilization")
plt.xticks(rotation=10)
plt.tight_layout()
plt.show()

## Part 4 — Financialize the price risk

A long-term compute reservation with a customer would be a **commercial supply contract**. That is *not* what we model here.

Instead, suppose a financial market lists a **cash-settled H100 compute forward/future** referenced to OCPI. You continue renting your GPUs in the physical market. Separately, you take a **short financial position** to hedge falling H100 prices. No GPU-hours are delivered through the derivative.

This now exists as a listed market: ICE Futures U.S. lists a monthly **cash-settled OCPI H100 future**. The contract settles against an average of OCPI H100 prices over the contract month; the first listed expiries began trading on October 5, 2026.

For simplicity, define:

- $K$ = the forward price agreed today; in this lab set $K=S_0$, today's latest OCPI value.
- $Q_F$ = financial contract notional in GPU-hours; set $Q_F=\hat{Q}=12{,}096$.
- $\bar{S}_T$ = the compute-price benchmark used for settlement over the contract period.

For the **short** forward position, cash settlement is

$$
\Pi_{\text{forward}} = Q_F\left(K-\bar{S}_T\right).
$$

If you physically rent exactly $Q_F$ GPU-hours at the same benchmark price, rental revenue plus forward P&L is

$$
R_{\text{combined}}
= Q_F\bar{S}_T + Q_F\left(K-\bar{S}_T\right)
= Q_FK.
$$

So the derivative hedges the **price per GPU-hour** without requiring the financial buyer to consume any compute.

For the scenario experiment below, treat the low/median/high OCPI values as illustrative settlement-benchmark outcomes. This is a mechanics exercise, not a forecast or a live Ornn forward quote.


In [ ]:
K = latest_price
Q_FORWARD = Q_EXPECTED
S_SETTLE = scenario_prices.copy()

# Physical rentals and the financial derivative are separate transactions.
spot_revenue = Q_FORWARD * S_SETTLE
forward_pnl = Q_FORWARD * (K - S_SETTLE)
hedged_revenue = spot_revenue + forward_pnl

hedge_table = pd.DataFrame({
    "settlement_benchmark_$/GPU-hour": S_SETTLE,
    "unhedged_physical_revenue_$": spot_revenue,
    "short_forward_PnL_$": forward_pnl,
    "combined_hedged_revenue_$": hedged_revenue,
})

display(hedge_table)

comparison = hedge_table[["unhedged_physical_revenue_$", "combined_hedged_revenue_$"]].copy()
comparison.columns = ["Unhedged physical rentals", "Physical rentals + short forward"]

plt.figure(figsize=(9, 4.5))
comparison.plot(kind="bar", ax=plt.gca())
plt.ylabel("90-day operator revenue ($)")
plt.xlabel("Settlement benchmark scenario")
plt.title("Price hedge: unhedged vs. cash-settled financial forward")
plt.xticks(rotation=10)
plt.legend()
plt.tight_layout()
plt.show()


### Question 2 — What did the financial contract do?

The combined hedged revenue is approximately the same across price scenarios **when the physical quantity sold equals the financial notional**.

Did the short forward create extra revenue, or did it change which risk the operator bears?

**Your answer:**  
`WRITE HERE`

## Part 5 — The price hedge does not eliminate utilization risk

The financial forward has a **fixed notional** \(Q_F\), chosen using the operator's expected 70% utilization. It does not reserve or rent the physical GPUs. If actual utilization turns out to be different, the operator's actual rental volume \(Q\) changes while the derivative notional stays fixed:

$$
R_{\text{hedged}}
=
Q\bar{S}_T + Q_F\left(K-\bar{S}_T\right).
$$

Equivalently,

$$
R_{\text{hedged}}
=
Q_FK + \left(Q-Q_F\right)\bar{S}_T.
$$

The bar chart below plots **total hedged operator revenue** under three actual-utilization levels and three settlement-price scenarios.

- At **70% utilization**, \(Q=Q_F\). The financial hedge exactly matches physical rental volume, so all three bars should be the same height.
- At **40% utilization**, \(Q<Q_F\). The operator is **over-hedged**.
- At **90% utilization**, \(Q>Q_F\). The operator is **under-hedged**.

The bars within 40% and 90% may still look fairly close because only the **mismatched quantity** is exposed to the settlement price.

> **Contrast:** in a take-or-pay physical reservation, a customer commits to pay for reserved capacity whether or not it uses it. That can transfer unused-capacity risk to the buyer. A cash-settled financial forward does not.


In [ ]:
utilizations = [0.40, 0.70, 0.90]
rows = []

# Physical rental volume changes with actual utilization, but the financial
# forward notional stays fixed at the quantity expected under 70% utilization.
for utilization in utilizations:
    actual_q = Q_MAX * utilization

    for scenario_name, settlement_price in scenario_prices.items():
        physical_revenue = actual_q * settlement_price
        forward_pnl = Q_FORWARD * (K - settlement_price)
        total_hedged_revenue = physical_revenue + forward_pnl

        rows.append({
            "actual_utilization": utilization,
            "price_scenario": scenario_name,
            "actual_GPU_hours": actual_q,
            "hedged_operator_revenue_$": total_hedged_revenue,
        })

utilization_table = pd.DataFrame(rows)

revenue_pivot = utilization_table.pivot(
    index="actual_utilization",
    columns="price_scenario",
    values="hedged_operator_revenue_$",
)
revenue_pivot.index = [f"{u:.0%}" for u in revenue_pivot.index]

display(revenue_pivot)

plt.figure(figsize=(9, 4.5))
revenue_pivot.plot(kind="bar", ax=plt.gca())
plt.ylabel("Total hedged operator revenue ($)")
plt.xlabel("Actual fleet utilization")
plt.title("Cash-settled forward: total revenue when utilization differs from forecast")
plt.xticks(rotation=0)
plt.legend(title="Settlement benchmark")
plt.tight_layout()
plt.show()


## Part 6 — Why standardization matters

A financial contract works best when the underlying service is credibly comparable across suppliers.

Consider two machines both advertised as `H100 SXM`:

| Attribute | Provider A | Provider B |
|---|---|---|
| Claimed GPU | H100 SXM | H100 SXM |
| Availability | 99.9% | 91% |
| GPU model / VRAM / CUDA | verified | self-reported |
| Network / interconnect | high performance | unknown |
| Service delivery | measured | weak evidence |

Ornn's published methodology describes verification of attributes including GPU specifications, CUDA version, VRAM, reliability, network throughput, system resources, and interconnect/PCIe bandwidth.

**Quick discussion:** What has to be verified before a physical GPU-hour can support a standardized benchmark and a cash-settled derivative?

Now move one layer up the stack: users usually do not want a GPU-hour itself. They want **inference**.


## Part 7 — Query a decentralized inference market

Darkbloom is a decentralized inference network that routes inference requests to verified Apple Silicon machines. Consumers pay model-specific token prices.

Darkbloom publishes current model pricing through a public endpoint. We will use **output-token pricing only** to keep the financial experiment focused.

Run the next cell to query the live pricing feed. The resulting table assigns each model a `model_id`; you will choose one of those models in Part 8.


In [ ]:
darkbloom_url = f"{DARKBLOOM_API_BASE}/v1/pricing"

db_response = requests.get(darkbloom_url, timeout=20)

if db_response.status_code == 429:
    raise RuntimeError("Darkbloom API rate limit reached. Ask the instructor to rerun later.")
db_response.raise_for_status()

db_payload = db_response.json()

raw_db_prices = db_payload.get("prices")
if not isinstance(raw_db_prices, list) or len(raw_db_prices) == 0:
    raise ValueError("Expected Darkbloom /v1/pricing to return a non-empty 'prices' list.")

darkbloom_prices = pd.DataFrame(raw_db_prices)

required_db_cols = {"model", "input_usd", "output_usd"}
missing = required_db_cols - set(darkbloom_prices.columns)
if missing:
    raise ValueError(f"Missing expected fields from Darkbloom response: {sorted(missing)}")

def parse_usd_rate(value):
    return float(str(value).replace("$", "").replace(",", "").strip())

darkbloom_prices["input_$/M_tokens"] = darkbloom_prices["input_usd"].map(parse_usd_rate)
darkbloom_prices["output_$/M_tokens"] = darkbloom_prices["output_usd"].map(parse_usd_rate)

if (darkbloom_prices[["input_$/M_tokens", "output_$/M_tokens"]] < 0).any().any():
    raise ValueError("Darkbloom token prices should not be negative.")

model_menu = (
    darkbloom_prices[
        ["model", "input_$/M_tokens", "output_$/M_tokens"]
    ]
    .sort_values("output_$/M_tokens")
    .reset_index(drop=True)
)
model_menu.index.name = "model_id"

display(model_menu)


## Part 8 — The inference intermediary game

You now run **Princeton Inference Co.**

Your company does **not** own the model-serving hardware. Instead:

1. You quote an enterprise customer a fixed price for next month's output inference.
2. You fulfill the contract by buying inference from Darkbloom at the market price then prevailing.
3. You can use an H100 compute forward as an **imperfect proxy hedge** against rising AI-infrastructure costs.
4. **You must choose your strategy before the future market state is revealed.**

Let:

- \(P_{D,0}\) = today's Darkbloom output-token price, in USD per million output tokens.
- \(P_C\) = the fixed customer price you quote today.
- \(K\) = today's H100 OCPI price, reused as the simplified compute-forward strike. This is a mechanics assumption, **not** a live ICE futures quote.

Ignoring input/cache charges,

$$
M_{\text{unhedged}}
=
Q_TP_C - Q_TP_{D,T}.
$$

A long H100 proxy hedge adds

$$
\Pi_H = Q_H(S_T-K).
$$

so

$$
M_{\text{hedged}}
=
M_{\text{unhedged}}+\Pi_H.
$$

Darkbloom runs heterogeneous Apple Silicon, not H100s. The H100 position is therefore only a **proxy hedge**: it may help when broad compute and inference prices move together, but it can fail when they diverge.

### TODO 1 — Commit your strategy

Change only the three values at the top of the next cell:

- **`MODEL_ID`** — choose any `model_id` from the live Darkbloom table above.
- **`CUSTOMER_MARKUP`** — choose `0.10`, `0.15`, or `0.25`.
- **`HEDGE_RATIO`** — choose `0.0`, `0.5`, `1.0`, or `1.5`.

Your markup affects how much business you win:

| Customer markup | Contract won |
|---:|---:|
| 10% | 1.5 billion output tokens |
| 15% | 1.0 billion output tokens |
| 25% | 0.6 billion output tokens |

These choices are calibrated so that, **at today's Darkbloom price**, all three markup choices start with the same expected dollar margin. A lower markup wins more volume but creates more inference-cost exposure.

`HEDGE_RATIO = 1.0` means the current dollar notional of the H100 hedge equals your expected Darkbloom inference spend. `0.5` is a half hedge; `1.5` intentionally over-hedges.

After choosing your strategy, run the cell **once**. The market will reveal one random future world. Submit the generated result.


In [ ]:
# TODO 1 — Commit your strategy BEFORE running the market.
MODEL_ID =                # choose any model_id from the table above
CUSTOMER_MARKUP =     # choose 0.10, 0.15, or 0.25
HEDGE_RATIO =         # choose 0.0, 0.5, 1.0, or 1.5

if MODEL_ID not in model_menu.index:
    raise ValueError(f"MODEL_ID must be one of {list(model_menu.index)}")
if CUSTOMER_MARKUP not in {0.10, 0.15, 0.25}:
    raise ValueError("CUSTOMER_MARKUP must be 0.10, 0.15, or 0.25.")
if HEDGE_RATIO not in {0.0, 0.5, 1.0, 1.5}:
    raise ValueError("HEDGE_RATIO must be 0.0, 0.5, 1.0, or 1.5.")

target_row = model_menu.loc[MODEL_ID]
TARGET_MODEL = str(target_row["model"])
P_D0 = float(target_row["output_$/M_tokens"])

# Pricing tradeoff: lower markup wins more volume.
TOKEN_QTY_BY_MARKUP_M = {
    0.10: 1_500.0,   # 1.5B output tokens
    0.15: 1_000.0,   # 1.0B output tokens
    0.25:   600.0,   # 0.6B output tokens
}
TOKEN_QTY_M = TOKEN_QTY_BY_MARKUP_M[CUSTOMER_MARKUP]

P_CUSTOMER = P_D0 * (1 + CUSTOMER_MARKUP)
CUSTOMER_REVENUE = TOKEN_QTY_M * P_CUSTOMER

# Dollar-notional proxy hedge.
EXPECTED_INFERENCE_COST = TOKEN_QTY_M * P_D0
FULL_HEDGE_GPU_HOURS = EXPECTED_INFERENCE_COST / K
HEDGE_GPU_HOURS = HEDGE_RATIO * FULL_HEDGE_GPU_HOURS

# Illustrative future worlds. These are stress scenarios, not forecasts.
WORLD_STATES = [
    {
        "world": "Broad compute glut",
        "Darkbloom_multiplier": 0.80,
        "H100_multiplier": 0.80,
        "description": "Broad AI-infrastructure prices fall together.",
        "weight": 0.15,
    },
    {
        "world": "Stable market",
        "Darkbloom_multiplier": 1.00,
        "H100_multiplier": 1.00,
        "description": "Both markets remain close to today's levels.",
        "weight": 0.10,
    },
    {
        "world": "Broad compute squeeze",
        "Darkbloom_multiplier": 1.30,
        "H100_multiplier": 1.30,
        "description": "Broad AI-infrastructure prices rise together.",
        "weight": 0.20,
    },
    {
        "world": "Mild AI boom",
        "Darkbloom_multiplier": 1.15,
        "H100_multiplier": 1.10,
        "description": "Inference and raw compute both rise, but not one-for-one.",
        "weight": 0.15,
    },
    {
        "world": "Darkbloom-specific shortage",
        "Darkbloom_multiplier": 1.35,
        "H100_multiplier": 1.00,
        "description": "Darkbloom inference gets more expensive while H100 stays flat.",
        "weight": 0.20,
    },
    {
        "world": "H100-specific shortage",
        "Darkbloom_multiplier": 1.05,
        "H100_multiplier": 1.40,
        "description": "H100 prices jump much more than Darkbloom inference prices.",
        "weight": 0.20,
    },
]

# A fresh Game ID creates a reproducible random world for this run.
import secrets
import random

GAME_ID = 100_000 + secrets.randbelow(900_000)
rng = random.Random(GAME_ID)
world = rng.choices(
    WORLD_STATES,
    weights=[w["weight"] for w in WORLD_STATES],
    k=1,
)[0]

P_DT = P_D0 * world["Darkbloom_multiplier"]
S_T = K * world["H100_multiplier"]

INFERENCE_COST = TOKEN_QTY_M * P_DT
UNHEDGED_MARGIN = CUSTOMER_REVENUE - INFERENCE_COST
FORWARD_PNL = HEDGE_GPU_HOURS * (S_T - K)
HEDGED_MARGIN = UNHEDGED_MARGIN + FORWARD_PNL
HEDGE_IMPACT = HEDGED_MARGIN - UNHEDGED_MARGIN

print("=" * 68)
print("PRINCETON INFERENCE CO. — MARKET REVEAL")
print("=" * 68)
print(f"Game ID: {GAME_ID}")
print()
print("YOUR STRATEGY")
print(f"  Model: {TARGET_MODEL}")
print(f"  Customer markup: {CUSTOMER_MARKUP:.0%}")
print(f"  Contract won: {TOKEN_QTY_M / 1_000:.1f}B output tokens")
print(f"  H100 hedge ratio: {HEDGE_RATIO:.0%}")
print()
print("THE WORLD")
print(f"  {world['world']}")
print(f"  {world['description']}")
print(f"  Darkbloom inference price: {(world['Darkbloom_multiplier'] - 1):+.0%}")
print(f"  H100 benchmark: {(world['H100_multiplier'] - 1):+.0%}")
print()
print("RESULT")
print(f"  Fixed customer revenue: ${CUSTOMER_REVENUE:,.2f}")
print(f"  Realized inference cost: ${INFERENCE_COST:,.2f}")
print(f"  Unhedged margin: ${UNHEDGED_MARGIN:,.2f}")
print(f"  H100 forward P&L: ${FORWARD_PNL:,.2f}")
print(f"  FINAL HEDGED MARGIN: ${HEDGED_MARGIN:,.2f}")
print("=" * 68)

plt.figure(figsize=(7.5, 4.2))
plt.bar(
    ["Unhedged margin", "H100-hedged margin"],
    [UNHEDGED_MARGIN, HEDGED_MARGIN],
)
plt.axhline(0, linewidth=1)
plt.ylabel("Intermediary margin ($)")
plt.title(f"Game {GAME_ID}: did the H100 proxy hedge help?")
plt.tight_layout()
plt.show()

SUBMISSION_OUTPUT = (
    f"Game {GAME_ID} | model={TARGET_MODEL} | markup={CUSTOMER_MARKUP:.0%} | "
    f"volume={TOKEN_QTY_M / 1_000:.1f}B tokens | hedge={HEDGE_RATIO:.0%} | "
    f"world={world['world']} | unhedged_margin=${UNHEDGED_MARGIN:,.2f} | "
    f"forward_PnL=${FORWARD_PNL:,.2f} | final_margin=${HEDGED_MARGIN:,.2f}"
)

print("\nSUBMIT THIS OUTPUT:")
print(SUBMISSION_OUTPUT)


## TODO 2 — Submit your game result

Submit the **`SUBMISSION_OUTPUT`** printed by the cell above, then answer in **2–3 sentences**:

1. Did your H100 hedge improve or worsen your realized margin?
2. Was the result mainly because of your hedge choice, or because Darkbloom inference and H100 prices moved differently?
3. Knowing the realized world, what would you change — and why could you not know that was the right choice **before** the world was revealed?

**Your answer:**  
`PASTE SUBMISSION_OUTPUT HERE`

`WRITE 2–3 SENTENCES HERE`


## Takeaway

The lab now has two layers of financialization.

First, raw compute can be standardized and financially hedged:

$$
\boxed{\text{physical infrastructure}}
\rightarrow
\boxed{\text{verified GPU-hour}}
\rightarrow
\boxed{\text{compute-price benchmark}}
\rightarrow
\boxed{\text{cash-settled compute derivative}}.
$$

Then an intermediary can transform a volatile infrastructure input into a fixed-price inference product:

$$
\boxed{\text{inference market}}
\rightarrow
\boxed{\text{fixed-price customer contract}}
\rightarrow
\boxed{\text{proxy hedge in compute futures}}.
$$

The proxy hedge works when the financial benchmark and the intermediary's actual costs move together. When they decouple, the intermediary is exposed to **basis risk**.

That suggests the next market to build: a standardized financial contract on **inference prices themselves**, one layer above raw compute.


---

### Data sources and course-use note

This notebook queries public, read-only price endpoints directly and does not package market data in the notebook source.

**Ornn**
- Public API documentation: https://index.ornn.com/docs
- OCPI methodology: https://data.ornn.com/methodology
- Compute-futures mechanics: https://ornn.com/insights/compute-futures
- ICE H100 futures: https://www.ice.com/products/83050006/Ornn-Compute-Price-Index-OCPI-H100-Future
- Terms: https://data.ornn.com/terms

**Darkbloom**
- Developer documentation: https://www.darkbloom.ai/developers
- Public pricing endpoint: https://api.darkbloom.dev/v1/pricing
- About / network architecture: https://www.darkbloom.ai/about
- Terms: https://www.darkbloom.ai/terms

For course distribution, keep notebook outputs cleared rather than packaging returned market data into the source notebook. Ornn's terms separately govern collection, redistribution, and publication/derived-data uses. Darkbloom is used here only through its public pricing endpoint; the lab does not send inference requests or require student credentials.
